# HSI Classification and Analysis Pipeline

This notebook performs RF classification, masking, and generates various visualization plots for HSI data analysis.

Run each section in order. This notebook is synchronized with the latest `classify.py` logic, including multi-granularity analysis, standardized bar plots, and prioritized S21 unit filtering.

## Setup and Imports

In [ ]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
import tifffile
import warnings
from glob import glob
import sys 
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from core.hsi_unlabeled_dataset import HSI_Unlabeled_Dataset
from core.hsi_visualizer import HSI_Visualizer
from core.hsi_sk_classifier import HSI_Classifier
from core.hsi_labeled_dataset import HSI_Labeled_Dataset
from config_loader import load_config, apply_config, list_available_profiles, print_config_info

## 1. Configuration Loading

In [ ]:
# CONFIGURATION PROFILE SELECTION
PROFILE = 'kidney_clusters'  

# Load configuration from selected profile
print(f"\nLoading configuration from profile: {PROFILE}")
config = load_config(PROFILE)
print_config_info(PROFILE)

# Apply configuration - unpack all settings
(perform_RF_classification, perform_masking, display_plots,
 mask_type, subgroups, display_units_maps, molecules_to_display, ratios_to_display,
 display_names, unit_mappings, unit_colors, unit_display_names, heatmap_unit_order,
 selected_molecules, selected_ratios, display_units_boxplots,
 individual_plot_config, sample_analysis_config) = apply_config(config)

# Extract individual plot settings early
create_individual_plots = individual_plot_config.get('create_individual_plots', True)
individual_molecules = individual_plot_config.get('selected_molecules', "All")
individual_ratios = individual_plot_config.get('selected_ratios', "All")
consolidate_Group_IDs = individual_plot_config.get('consolidate_Group_IDs', True)

# Extract granularity and sample analysis settings
granularity_levels = sample_analysis_config.get('granularity_levels', [])
selected_values = sample_analysis_config.get('selected_values', None)
filter_col = sample_analysis_config.get('filter_col', 'Source_ID') 

# Derive bubble_unit_order as reverse of heatmap order
bubble_unit_order = heatmap_unit_order[::-1]

print("\n✓ Configuration loaded successfully")

## 2. Dataset and Classifier Initialization

In [ ]:
workspace_dir = str(Path.cwd().parent)
base_directory = r"D:\integrated_pipeline\HSI_data\data"

# SRS parameters
wn_1, wn_2, num_samp = 2700, 3100, 61
ch_start = int(((wn_2 - 2800)/(wn_2 - wn_1)) * num_samp)

chosen_model_name = 'rf_n150_md40_mss2_msl9_a9_best_model_platt'
model_path = os.path.join(workspace_dir, 'sk_classifier', 'models', f'{chosen_model_name}.joblib')

# Create visualizer
visualizer = HSI_Visualizer(
    mol_path=os.path.join(workspace_dir, 'molecule_dataset', 'lipid_subtype_wn_61_test.npz'),
    wavenumber_start=wn_1, wavenumber_end=wn_2, num_samples=num_samp
)
plots_directory = os.path.join(os.path.dirname(base_directory), f'{chosen_model_name}_{mask_type}_plots')

# Create dataset
dataset = HSI_Unlabeled_Dataset(
    base_directory, ch_start, image_normalization=True, min_max_normalization=False,
    num_samples=num_samp, wavenumber_start=wn_1, wavenumber_end=wn_2
)

labeled_dataset = HSI_Labeled_Dataset(
    molecule_dataset_path=os.path.join(workspace_dir, 'molecule_dataset', 'lipid_subtype_wn_61_test'),
    srs_params_path=os.path.join(workspace_dir, 'params_dataset', 'srs_params_61'),
    num_samples_per_class=10000, compute_min_max=True
)

# Initialize classifier
output_base = os.path.join(os.path.dirname(base_directory), f"{chosen_model_name}_outputs")
classifier = HSI_Classifier(dataset, model_path=model_path, output_base=output_base, 
                            visualizer=visualizer, labeled_dataset=labeled_dataset)

percentages_df, ratios_df = None, None
print("✓ Dataset and classifier initialized successfully")

## 3. Classification/Prediction

In [ ]:
if perform_RF_classification:
    if os.path.exists(model_path):
        print("Running RandomForest inference...")
        classifier.predict(alpha=10, generate_shap=True)
    else:
        print(f"No model found at {model_path}")
else:
    print("RF classification disabled")

## 4. Masking and Quantification

In [ ]:
masked_output_dir = os.path.join(os.path.dirname(base_directory), f"{mask_type}_output_csv")
os.makedirs(masked_output_dir, exist_ok=True)
output_csv = os.path.join(masked_output_dir, 'masked_predictions_percentages.csv')
output_ratio_csv = os.path.join(masked_output_dir, 'masked_ratio_means.csv')

if perform_masking:
    mask_prefix = mask_type.split("_")[0]  
    print("Processing masks and quantifying results...")
    
    predictions_per_unit, ratios_per_unit = {}, {}
    
    for img_idx, img_path in tqdm(enumerate(dataset.img_list), total=len(dataset.img_list)):
        img_name_no_ext = os.path.splitext(os.path.basename(img_path))[0]
        mask_folder = os.path.join(os.path.dirname(base_directory), mask_type, img_name_no_ext)
        csv_path = os.path.join(classifier.output_base, img_name_no_ext, f"{img_name_no_ext}_predictions.csv")
        ratio_folder = os.path.join(os.path.dirname(base_directory), 'Ratio', img_name_no_ext)
        
        if not os.path.exists(mask_folder): continue

        if os.path.exists(csv_path):
            res = visualizer.apply_rf_masking(csv_path, mask_folder, subgroups, img_name_no_ext, mask_prefix, group_subclasses=True)
            for u, e in res.items(): predictions_per_unit.setdefault(u, []).extend(e)
        
        if os.path.exists(ratio_folder):
            for r_tif in glob(os.path.join(ratio_folder, '*.tif')):
                if '_masked' in r_tif: continue
                res = visualizer.apply_rf_masking(ratio_tiff_path=r_tif, mask_list_path=mask_folder, subgroups=subgroups, img_name=img_name_no_ext, prefix=mask_prefix)
                for u, e in res.items(): ratios_per_unit.setdefault(u, []).extend(e)
    
    if predictions_per_unit:
        percentages_df = visualizer.quantify_unit_class_percentages_nested(predictions_per_unit, unit_mappings=unit_mappings)
        percentages_df.to_csv(output_csv, index=False)
    
    if ratios_per_unit:
        ratios_df = visualizer.quantify_unit_ratio_means_nested(ratios_per_unit, unit_mappings=unit_mappings)
        ratios_df.to_csv(output_ratio_csv, index=False)
else:
    print("Masking disabled")

## 5. Visualization Setup and Prioritized Filtering

In [ ]:
if display_plots:
    if percentages_df is None: percentages_df = pd.read_csv(output_csv)
    if ratios_df is None and os.path.exists(output_ratio_csv): ratios_df = pd.read_csv(output_ratio_csv)
        
    # Apply global filters
    if selected_values is not None and filter_col:
        for df in [percentages_df, ratios_df]:
            if df is not None and filter_col in df.columns:
                df = df[df[filter_col].isin(selected_values)].copy()
    
    # --- S21-Anchored Unit Filtering ---
    anchor_source = 'S21'
    if percentages_df is not None and 'Source_ID' in percentages_df.columns:
        s21_units = percentages_df[percentages_df['Source_ID'] == anchor_source]['Unit'].unique()
        
        if len(s21_units) > 0:
            # Anchor units (keep units found in S21, include their S24 data if it exists)
            percentages_df = percentages_df[percentages_df['Unit'].isin(s21_units)].copy()
            print(f"  Applied unit-level filter: Kept {len(s21_units)} units present in {anchor_source} (percentages)")
            
            if ratios_df is not None:
                # 1. Prioritize S21 for Redox (Exclude S24 Redox)
                redox_mask = (ratios_df['Ratio_Type'] == 'Redox') & (ratios_df['Source_ID'] != anchor_source)
                ratios_df = ratios_df[~redox_mask].copy()
                
                # 2. Re-anchor Ratio units
                ratios_df = ratios_df[ratios_df['Unit'].isin(s21_units)].copy()
                print(f"  Applied unit-level filter: Kept units present in {anchor_source} for ratios")
                print(f"  Applied item-level filter: Redox analysis restricted to {anchor_source} only")
        else:
             print(f"  Warning: No data found for anchor source {anchor_source}.")
        
    print("✓ Data filtered and prepared for plotting")

## 6. Multi-Granularity Analysis Loop

Generates heatmaps, bubble charts, and individual bar plots for each defined granularity level (e.g., Source_ID, Group_ID).

In [ ]:
if display_plots and granularity_levels:
    print("GENERATING GRANULARITY-SPECIFIC PLOTS")
    
    for group_cols in granularity_levels:
        cols_to_use = [c for c in group_cols if c in (percentages_df.columns if percentages_df is not None else []) or c in (ratios_df.columns if ratios_df is not None else [])]
        level_folder = "_".join(cols_to_use)
        
        for df_label, df_iter, val_col, grp_col, dtype in [
            ('percentages', percentages_df, 'Percentage', 'Molecule', 'percentage'),
            ('ratios', ratios_df, 'Mean_Ratio', 'Ratio_Type', 'ratio'),
        ]:
            if df_iter is None or df_iter.empty: continue
            if not all(col in df_iter.columns for col in cols_to_use): continue
            
            unique_combos = df_iter[cols_to_use].drop_duplicates().values
            for combo in unique_combos:
                mask = True
                combo_parts = []
                for i, col in enumerate(cols_to_use):
                    mask &= (df_iter[col] == combo[i])
                    combo_parts.append(str(combo[i]))
                
                subset_df = df_iter[mask].copy()
                combo_str = "_".join(combo_parts)
                output_dir_base = os.path.join(plots_directory, f'granularity_{df_label}', level_folder, combo_str)
                _groups = molecules_to_display if grp_col == 'Molecule' else ratios_to_display
                
                visualizer.generate_unit_heatmaps(subset_df, val_col, grp_col, os.path.join(output_dir_base, 'heatmaps'), 
                                                  dtype, groups_to_display=_groups, display_name_map=display_names, 
                                                  unit_display_map=unit_display_names, unit_order=heatmap_unit_order)
                
                visualizer.generate_unit_bubble_charts(subset_df, val_col, grp_col, os.path.join(output_dir_base, 'bubble_plots'),
                                                       dtype, groups_to_display=_groups, display_name_map=display_names,
                                                       unit_display_map=unit_display_names, unit_order=bubble_unit_order)
                
                if create_individual_plots and len(cols_to_use) == 1:
                    bar_items = individual_molecules if grp_col == 'Molecule' else individual_ratios
                    visualizer.create_individual_barplots(
                        df=subset_df, value_col=val_col, grouping_col=grp_col, items_list=bar_items,
                        output_dir=os.path.join(output_dir_base, f'individual_bar_{display_units_boxplots_type}_plots'), data_type=dtype,
                        display_name_map=display_names, unit_color_map=unit_colors, unit_display_map=unit_display_names,
                        units_to_display=display_units_boxplots, compare_by=None, consolidate_Group_IDs=consolidate_Group_IDs
                    )
    print("✓ Granularity-specific analysis completed")

## 7. Global Summary Heatmaps and Bubble Charts

In [ ]:
if display_plots:
    print("GENERATING GLOBAL SUMMARY PLOTS")
    
    for df_label, df_iter, val_col, grp_col, dtype in [
        ('percentages', percentages_df, 'Percentage', 'Molecule', 'percentage'),
        ('ratios', ratios_df, 'Mean_Ratio', 'Ratio_Type', 'ratio'),
    ]:
        if df_iter is None or df_iter.empty: continue
        _groups = molecules_to_display if grp_col == 'Molecule' else ratios_to_display
        
        visualizer.generate_unit_heatmaps(df_iter, val_col, grp_col, os.path.join(plots_directory, f'heatmaps_global_{df_label}'),
                                          dtype, groups_to_display=_groups, display_name_map=display_names,
                                          unit_display_map=unit_display_names, unit_order=heatmap_unit_order)
        
        visualizer.generate_unit_bubble_charts(df_iter, val_col, grp_col, os.path.join(plots_directory, f'bubble_plots_global_{df_label}'),
                                               dtype, groups_to_display=_groups, display_name_map=display_names,
                                               unit_display_map=unit_display_names, unit_order=bubble_unit_order)
    
    print("✓ Global summary plots completed")